In [2]:
import torch
from transformers import AutoModel, AutoProcessor
from tqdm import tqdm

ckpt = "google/siglip2-base-patch16-naflex"

# Show download/load progress
with tqdm(total=2, desc="Loading model") as pbar:
    full_model = AutoModel.from_pretrained(ckpt, device_map="auto")
    pbar.update(1)
    pbar.set_description("Loading processor")
    
    processor = AutoProcessor.from_pretrained(ckpt)
    pbar.update(1)
    pbar.set_description("Done")

# Extract ONLY the image encoder (vision tower) with its pretrained weights
print("Extracting vision tower...", end=" ")
vision_encoder = full_model.vision_model
print("✓")

# Free the text tower from memory — you don't need it
print("Freeing text tower from memory...", end=" ")
del full_model
torch.cuda.empty_cache()
print("✓")

print(f"\nVision encoder ready — hidden size: {vision_encoder.config.hidden_size}")

Done: 100%|██████████| 2/2 [12:23<00:00, 371.59s/it]             

Extracting vision tower... ✓
Freeing text tower from memory... ✓

Vision encoder ready — hidden size: 1152


In [ ]:
import torch.nn as nn

class SigLIP2ImageEncoder(nn.Module):
    def __init__(self, vision_encoder):
        super().__init__()
        self.encoder = vision_encoder

    def forward(self, pixel_values, pixel_mask=None):
        outputs = self.encoder(pixel_values=pixel_values, pixel_mask=pixel_mask)
        return nn.functional.normalize(outputs.pooler_output, dim=-1)

model = SigLIP2ImageEncoder(vision_encoder).cuda()

In [4]:
from PIL import Image

def load_batch(image_paths):
    images = [Image.open(p).convert("RGB") for p in image_paths]
    # NaFlex processor handles variable resolution automatically
    inputs = processor(images=images, return_tensors="pt")
    return {k: v.cuda() for k, v in inputs.items()}

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)

# Example: contrastive loss, triplet loss, or simple MSE — your choice
for batch in your_dataloader:
    inputs = load_batch(batch["image_paths"])
    embeddings = model(**inputs)
    
    loss = your_loss_fn(embeddings, batch["labels"])
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()